# Weather-related flight disruption prediction with AutoGluon

Predict `weather_disruption` from flight schedules and weather observations.

| Split | Departure period (UTC) | Use |
| --- | --- | --- |
| Training | 2019–2022 | Fit models |
| Validation | 2023 | Select models |
| Calibration | 2024 | Select decision threshold |
| Test | 2025 (available records) | Final evaluation |

## 1. Select airports

Set `ORIGIN_AIRPORTS` to one or more IATA code. Name each experiment using `EXPERIMENT_NAME`. Date ranges use UTC and exclude the end date.

Results are saved to `data/<EXPERIMENT_NAME>/results.json`; the AutoGluon predictor and its models are saved to `data/<EXPERIMENT_NAME>/model/`. Use a distinct experiment name for each run.

In [ ]:
from pathlib import Path
import json
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display
from autogluon.tabular import TabularDataset, TabularPredictor


ORIGIN_AIRPORTS = ['DFW']  # E.g. ['DFW', 'ORD']; None for all airports.
EXPERIMENT_NAME = '1_airport'

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Start in the project root or notebooks directory.'


if (not isinstance(EXPERIMENT_NAME, str) or not EXPERIMENT_NAME.strip()
        or EXPERIMENT_NAME in {'.', '..'}
        or '/' in EXPERIMENT_NAME or '\\' in EXPERIMENT_NAME):
    raise ValueError('EXPERIMENT_NAME must be a non-empty directory name.')
EXPERIMENT_DIR = ROOT / 'data' / EXPERIMENT_NAME
MODEL_DIR = EXPERIMENT_DIR / 'model'
RESULTS_PATH = EXPERIMENT_DIR / 'results.json'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)


if ORIGIN_AIRPORTS is not None:
    if not isinstance(ORIGIN_AIRPORTS, list) or not ORIGIN_AIRPORTS:
        raise ValueError('ORIGIN_AIRPORTS must be a non-empty list or None.')
    if not all(isinstance(airport, str) and airport.strip() for airport in ORIGIN_AIRPORTS):
        raise ValueError('Each airport must be a non-empty string.')
    ORIGIN_AIRPORTS = list(dict.fromkeys(airport.strip().upper() for airport in ORIGIN_AIRPORTS))
PERIODS = {
    'training': ('2019-01-01', '2023-01-01'),
    'validation': ('2023-01-01', '2024-01-01'),
    'calibration': ('2024-01-01', '2025-01-01'),
    'test': ('2025-01-01', '2026-01-01'),
}

previous_end = None
for name, (start, end) in PERIODS.items():
    assert pd.Timestamp(start) < pd.Timestamp(end), f'{name}: start must precede end.'
    if previous_end is not None:
        assert pd.Timestamp(previous_end) <= pd.Timestamp(start), 'Periods must not overlap.'
    previous_end = end

DATA = ROOT / 'data' / 'bts_flights_weather' / 'flights_weather'
files = sorted(DATA.glob('year=*/*.parquet'))
assert files, 'Download the BTS flight/weather year partitions first.'
con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET TimeZone = 'UTC'")
con.execute("SET memory_limit = '2GB'")
con.read_parquet([str(file) for file in files], hive_partitioning=False).create_view('flights')
print('Origin airports:', ORIGIN_AIRPORTS if ORIGIN_AIRPORTS is not None else 'all origins')
print('Available year folders:', sorted({file.parent.name for file in files}))

## 2. Prepare features and target

`weather_disruption = 1` for a weather-related cancellation, or when **both** reported weather delay and departure delay exceed 15 minutes.

In [ ]:
# Define features and construct the target.

TARGET = 'weather_disruption'
OUTCOMES = ['reported_weather_delay_min', 'dep_delay_min', 'cancelled', 'cancellation_reason']
GROUPS = ['wx_origin', 'wx_origin_m1', 'wx_origin_m3', 'wx_origin_m6', 'wx_dest', 'wx_dest_m3']
# Approved predictors.
SCHEDULE_FEATURES = ['operating_carrier', 'origin_iata', 'dest_iata', 'distance_miles', 'crs_elapsed_min', 'day_of_week', 'day_of_year', 'month', 'is_us_federal_holiday', 'days_to_nearest_holiday', 'is_holiday_travel_window']
WEATHER_FEATURES = ['wx_origin_temp_c', 'wx_origin_dewpoint_c', 'wx_origin_relative_humidity_pct', 'wx_origin_wind_speed_ms', 'wx_origin_wind_gust_ms', 'wx_origin_wind_dir_deg', 'wx_origin_visibility_m', 'wx_origin_ceiling_m', 'wx_origin_ceiling_unlimited', 'wx_origin_sea_level_pressure_hpa', 'wx_origin_precip_1h_mm', 'wx_origin_snow_depth_cm', 'wx_origin_sky_cover_oktas', 'wx_origin_thunderstorm', 'wx_origin_snow', 'wx_origin_freezing_precip', 'wx_origin_fog', 'wx_origin_rain', 'wx_origin_flight_category', 'wx_origin_flight_category_ordinal', 'wx_origin_obs_missing', 'wx_origin_m1_temp_c', 'wx_origin_m1_wind_speed_ms', 'wx_origin_m1_wind_gust_ms', 'wx_origin_m1_visibility_m', 'wx_origin_m1_ceiling_m', 'wx_origin_m1_precip_1h_mm', 'wx_origin_m1_flight_category_ordinal', 'wx_origin_m1_thunderstorm', 'wx_origin_m1_snow', 'wx_origin_m1_freezing_precip', 'wx_origin_m1_fog', 'wx_origin_m1_rain', 'wx_origin_m1_obs_missing', 'wx_origin_m3_temp_c', 'wx_origin_m3_wind_speed_ms', 'wx_origin_m3_wind_gust_ms', 'wx_origin_m3_visibility_m', 'wx_origin_m3_ceiling_m', 'wx_origin_m3_precip_1h_mm', 'wx_origin_m3_flight_category_ordinal', 'wx_origin_m3_thunderstorm', 'wx_origin_m3_snow', 'wx_origin_m3_freezing_precip', 'wx_origin_m3_fog', 'wx_origin_m3_rain', 'wx_origin_m3_obs_missing', 'wx_origin_m6_temp_c', 'wx_origin_m6_wind_speed_ms', 'wx_origin_m6_wind_gust_ms', 'wx_origin_m6_visibility_m', 'wx_origin_m6_ceiling_m', 'wx_origin_m6_precip_1h_mm', 'wx_origin_m6_flight_category_ordinal', 'wx_origin_m6_thunderstorm', 'wx_origin_m6_snow', 'wx_origin_m6_freezing_precip', 'wx_origin_m6_fog', 'wx_origin_m6_rain', 'wx_origin_m6_obs_missing', 'wx_dest_temp_c', 'wx_dest_dewpoint_c', 'wx_dest_relative_humidity_pct', 'wx_dest_wind_speed_ms', 'wx_dest_wind_gust_ms', 'wx_dest_wind_dir_deg', 'wx_dest_visibility_m', 'wx_dest_ceiling_m', 'wx_dest_ceiling_unlimited', 'wx_dest_sea_level_pressure_hpa', 'wx_dest_precip_1h_mm', 'wx_dest_snow_depth_cm', 'wx_dest_sky_cover_oktas', 'wx_dest_thunderstorm', 'wx_dest_snow', 'wx_dest_freezing_precip', 'wx_dest_fog', 'wx_dest_rain', 'wx_dest_flight_category', 'wx_dest_flight_category_ordinal', 'wx_dest_obs_missing', 'wx_dest_m3_temp_c', 'wx_dest_m3_wind_speed_ms', 'wx_dest_m3_wind_gust_ms', 'wx_dest_m3_visibility_m', 'wx_dest_m3_ceiling_m', 'wx_dest_m3_precip_1h_mm', 'wx_dest_m3_flight_category_ordinal', 'wx_dest_m3_thunderstorm', 'wx_dest_m3_snow', 'wx_dest_m3_freezing_precip', 'wx_dest_m3_fog', 'wx_dest_m3_rain', 'wx_dest_m3_obs_missing']
FEATURES = SCHEDULE_FEATURES + WEATHER_FEATURES
LOAD_COLUMNS = list(dict.fromkeys(
    FEATURES + OUTCOMES + ['info_cutoff_utc', 'sched_dep_utc']
    + [group + '_obs_time_utc' for group in GROUPS]
))

available_columns = con.execute("DESCRIBE flights").fetchdf()['column_name'].tolist()
missing_columns = set(LOAD_COLUMNS) - set(available_columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'
EXCLUDED_COLUMNS = sorted(set(available_columns) - set(FEATURES))
print('Predictor columns:', len(FEATURES))
print('Source columns excluded from model inputs:', EXCLUDED_COLUMNS)

def prepare_model_data(frame):
    """Create labels and exclude future weather observations."""
    cutoff = frame['info_cutoff_utc']
    assert cutoff.notna().all() and cutoff.le(frame['sched_dep_utc']).all(), (
        'Every row needs a cutoff no later than scheduled departure.'
    )
    frame[TARGET] = (
        ((frame['reported_weather_delay_min'] > 15) & (frame['dep_delay_min'] > 15))
        | (frame['cancelled'].fillna(False) & frame['cancellation_reason'].eq('Weather'))
    ).fillna(False).astype('int8')
    for group in GROUPS:
        stamp = frame[group + '_obs_time_utc']
        invalid = stamp.isna() | stamp.gt(cutoff)
        fields = [column for column in WEATHER_FEATURES
                  if column.startswith(group + '_') and not any(
                      column.startswith(other + '_') for other in GROUPS
                      if other.startswith(group + '_'))]
        frame.loc[invalid, fields] = pd.NA
    # Convert missing Arrow values to np.nan for AutoGluon.
    categorical_features = ['operating_carrier', 'origin_iata', 'dest_iata',
                            'wx_origin_flight_category', 'wx_dest_flight_category']
    result = {}
    for column in FEATURES:
        if column in categorical_features:
            values = frame[column].astype(object)
            result[column] = values.where(pd.notna(values), np.nan)
        else:
            result[column] = pd.to_numeric(frame[column], errors='raise').to_numpy(
                dtype=np.float32, na_value=np.nan
            )
    result[TARGET] = frame[TARGET].to_numpy(dtype=np.int8)
    return pd.DataFrame(result, index=frame.index)

def load_period(start, end):
    """Load one period with approved features and the target."""
    columns_sql = ', '.join('"' + column + '"' for column in LOAD_COLUMNS)
    result = con.sql(f"""
        SELECT {columns_sql}
        FROM flights
        WHERE sched_dep_utc >= CAST(? AS TIMESTAMPTZ)
          AND sched_dep_utc < CAST(? AS TIMESTAMPTZ)
          AND (? IS NULL OR list_contains(?, origin_iata))
        ORDER BY sched_dep_utc
    """, params=[start, end, ORIGIN_AIRPORTS, ORIGIN_AIRPORTS])
    frame = result.to_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
    return prepare_model_data(frame)


## 3. Load the data splits

Each split contains only the approved predictors and `TARGET`; outcome variables used to build the label are excluded from model inputs.

In [ ]:
train_data = load_period(*PERIODS['training'])
validation_data = load_period(*PERIODS['validation'])
calibration_data = load_period(*PERIODS['calibration'])
test_data = load_period(*PERIODS['test'])

## 4. Train AutoGluon

In [ ]:
# TODO: Convert all four splits to AutoGluon TabularDataset objects.

In [ ]:
# Docs: https://auto.gluon.ai/stable/tutorials/tabular/tabular-quick-start.html
# API: https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.fit.html
# Models: https://auto.gluon.ai/stable/api/autogluon.tabular.models.html

# TODO: Fit the predictor below, using average_precision as metric to optimize.
# Use validation_data as tuning_data; set time_limit in seconds.
# Start with included_model_types=['XGB']; then try RF and LR.

predictor = TabularPredictor(
    # .....
    # .....
    # .....
    path=str(MODEL_DIR),
)
# TODO: predictor.fit(...)

## 5. Compare models

Review the validation leaderboard. Higher `score_val` (average precision) is better.

In [ ]:
# TODO: Display the validation leaderboard.
# leaderboard = predictor.leaderboard(silent=True)
# display(leaderboard)

## 6. Calibrate the decision threshold (2024)

Choose the classification threshold using the calibration split only.

In [ ]:
# TODO: Calibrate the threshold with calibration_data (not test_data).
# https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.calibrate_decision_threshold.html

## 7. Evaluate on the test split (2025)

Report final metrics with the threshold chosen on 2024; do not tune on test data.

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             accuracy_score, confusion_matrix)
y_test = test_data[TARGET]
x_test = test_data.drop(columns=TARGET)
test_probabilities = predictor.predict_proba(x_test, as_multiclass=False)
test_predictions = predictor.predict(x_test)
assert np.isfinite(test_probabilities).all()
test_metrics = pd.Series({
    'flights': len(y_test),
    'positives': int(y_test.sum()),
    'positive_rate': float(y_test.mean()),
    'decision_threshold': predictor.decision_threshold,
    'average_precision': average_precision_score(y_test, test_probabilities),
    'roc_auc': roc_auc_score(y_test, test_probabilities),
    'precision': precision_score(y_test, test_predictions, zero_division=0),
    'recall': recall_score(y_test, test_predictions, zero_division=0),
    'f1': f1_score(y_test, test_predictions, zero_division=0),
    'accuracy': accuracy_score(y_test, test_predictions),
    'alert_rate': float((test_predictions == 1).mean()),
}, name='test')
test_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=['actual_negative', 'actual_positive'],
    columns=['predicted_negative', 'predicted_positive'],
)
print('Predictor:', predictor.path)
print('Model:', predictor.model_best)
print(test_metrics.to_string())
print(test_confusion_matrix.to_string())


In [ ]:
# Persist the calibrated predictor and final experiment results.
predictor.save()
experiment_results = {
    'experiment_name': EXPERIMENT_NAME,
    'origin_airports': ORIGIN_AIRPORTS,
    'periods': PERIODS,
    'target': TARGET,
    'features': FEATURES,
    'model_path': str(MODEL_DIR.relative_to(ROOT)),
    'best_model': predictor.model_best,
    'test_metrics': {
        key: int(value) if key in {'flights', 'positives'} else float(value)
        for key, value in test_metrics.items()
    },
    'test_confusion_matrix': test_confusion_matrix.to_dict(orient='index'),
    'validation_leaderboard': json.loads(
        predictor.leaderboard(silent=True).to_json(orient='records')
    ),
}
RESULTS_PATH.write_text(
    json.dumps(experiment_results, indent=2, allow_nan=False) + '\n',
    encoding='utf-8',
)
print('Saved results:', RESULTS_PATH)
print('Saved AutoGluon predictor:', MODEL_DIR)


## 8. Inspect feature importance

Compute [`predictor.feature_importance`](https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.feature_importance.html) and plot the top 20 features.

In [ ]:
# TODO: Compute feature_importance before plotting.
feature_importance = None

import matplotlib.pyplot as plt
plot_data = feature_importance.head(20).sort_values('importance')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_data.index, plot_data['importance'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Decrease in test average precision after permutation')
ax.set_title('Top 20 feature importances — test partition')
fig.tight_layout()
plt.show()


## Next steps

Using the **airport ranking from the previous notebook**, repeat this analysis for the top **5, 10, 15, and 20 airports** impacted by departure delay. Compare the test metrics across the four groups. Change experiment_name each time you try a new combintion.
Feel free to explore, try other combinations and do your own experiments.

GOAL: Identify the combination that maximizes recall metric